# Phase 2E — Question-Type Analysis

Which memory architecture wins on which *type* of question? The proposal defines 7 question types
(fact recall, multi-hop, temporal, pattern, failure, comparison, abstention). This notebook breaks
down the benchmark results by type × strategy, revealing that **no single architecture dominates
across all types** — the core benchmarking insight.

**Independent notebook** — runs standalone offline (no API key needed).

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "experiment" / "github_submission" / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

print("SOURCE_DIR =", SOURCE_DIR)

In [ ]:
# Load the raw benchmark trace
import json
from collections import defaultdict
raw = json.loads((RESULTS_DIR / 'metrics' / 'sample_real_raw.json').read_text())
records = raw['records']
print(f'Loaded {len(records)} records')
cats = sorted(set(r.get('category','unknown') for r in records))
print(f'Question types: {cats}')
strats = sorted(set(r.get('strategy','?') for r in records))
print(f'Strategies: {strats}')


## 📖 Narrative: Question-type diversity

The benchmark covers many question types: `single_hop` (direct fact lookup), `multi_hop`
(requires combining facts), `temporal_reasoning` (time-based), `knowledge-update`
(conflicting information across sessions). Different architectures may excel at different types.

In [ ]:
# Group by category × strategy; compute hit rate, precision, recall
print(f'{"type":>22} {"strategy":>16} {"n":>4} {"hit%":>6} {"P":>6} {"R":>6}')
print('-' * 65)
by_type = defaultdict(lambda: defaultdict(list))
for r in records:
    by_type[r.get('category','unknown')][r.get('strategy','?')].append(r)

for cat in sorted(by_type):
    for strat in ['no_memory','verbatim','extracted_facts','episodic','hybrid']:
        recs = by_type[cat].get(strat, [])
        if not recs:
            continue
        n = len(recs)
        hit = sum(1 for r in recs if r.get('evidence_hit')) / n
        prec = sum(r.get('retrieval_precision',0) for r in recs) / n
        rec = sum(r.get('retrieval_recall',0) for r in recs) / n
        print(f'{cat:>22} {strat:>16} {n:>4} {hit:>5.0%} {prec:>6.3f} {rec:>6.3f}')
    print()


## 📖 Narrative: The per-type breakdown

This table reveals which architecture wins on each question type. Look for patterns:
- Does **verbatim** win on `single_hop`? (Raw text = exact match)
- Does **episodic** win on `temporal_reasoning`? (Episodes preserve time context)
- Does **extracted_facts** win on `multi_hop`? (Atomic facts are combinable)

### 📝 Quick Quiz
1. **Which type has the most records?** Which has the highest hit rate overall?
2. **Find a type where extracted_facts beats verbatim.** Why might fact extraction help?
3. **Find a type where episodic wins.** What about episodes suits that type?
4. **Are there types where ALL providers score 0?** What makes those types hard?

In [ ]:
# Find the WINNER per question type (excluding no_memory)
print('Winner per question type (by evidence hit rate):')
print(f'{"type":>22} {"winner":>16} {"hit%":>6}')
print('-' * 48)
for cat in sorted(by_type):
    best_strat, best_hit = None, -1
    for strat in ['verbatim','extracted_facts','episodic','hybrid']:
        recs = by_type[cat].get(strat, [])
        if recs:
            hit = sum(1 for r in recs if r.get('evidence_hit')) / len(recs)
            if hit > best_hit:
                best_hit, best_strat = hit, strat
    if best_strat:
        print(f'{cat:>22} {best_strat:>16} {best_hit:>5.0%}')


## 📖 Narrative: No single winner

### 📝 Quick Quiz
1. **Is the same provider the winner on every type?** What does that tell you?
2. **If you could only use ONE provider, which would you choose?** Justify with data.
3. **The hybrid provider combines verbatim + episodic.** Does it win more types than either alone?
4. **Design a hypothetical "adaptive" system** that picks the provider per question type.
   How much improvement would it get over always using one?

### 💡 Bridge to Phase 3
This per-type analysis shows that **memory architecture choice depends on the workload**.
In Phase 3, we'll apply the same diagnostic thinking to autonomous research — where the
"questions" are experiment-config-selection decisions, and the "memory" is the experiment history.

## What this reveals

- **Different types have different winners.** Verbatim may win on fact-recall (exact-match);
 episodic may win on temporal reasoning (session summaries preserve time context); extracted facts
 may win on multi-hop (atomic facts are easier to combine).
- **No single architecture dominates across all types** — the core benchmarking message.
 This is why the proposal advocates a *pluggable, multi-provider* benchmark rather than declaring
 one 'best' memory system.
- **For your own application:** identify which question types matter most, then choose the
 architecture that wins on those types (or combine them via the hybrid provider).

**Connection to Phase 3:** the same diagnostic approach (decomposing *where* memory fails by type)
applies to autoresearch experiment memory — which is exactly what the Phase 3D memory diagnostic
does for OOM/regression failures.